# ORACLE: Gene Regulatory Network Inference

Infer and visualize the hybrid GRN combining GRNBoost2 data-driven inference
with TRRUST v2 curated priors and ENCODE ChIP-seq evidence.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import anndata as ad
from pathlib import Path

DATA_DIR = Path('../data')

## 1. Load Preprocessed AnnData

In [ ]:
adata = ad.read_h5ad(DATA_DIR / 'processed/anndata/colorectal_GSE132465_processed.h5ad')
print(f'Loaded {adata.n_obs} cells x {adata.n_vars} genes')

## 2. Run GRN Inference

In [ ]:
from oracle.cam.grn_inference import GRNInferenceEngine
from oracle.cam.preprocessing import CAMConfig

config = CAMConfig(cancer_type='colorectal', tissue='colon', grn_size=100)
engine = GRNInferenceEngine(config)
grn = engine.infer(adata)

print(f'GRN: {grn.number_of_nodes()} nodes, {grn.number_of_edges()} edges')
print(f'Activating edges: {sum(1 for _,_,d in grn.edges(data=True) if d.get("sign",1)>0)}')
print(f'Repressing edges: {sum(1 for _,_,d in grn.edges(data=True) if d.get("sign",1)<0)}')

## 3. Network Statistics

In [ ]:
degrees = dict(grn.degree())
in_degrees = dict(grn.in_degree())
out_degrees = dict(grn.out_degree())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(list(degrees.values()), bins=30, color='steelblue')
axes[0].set_title('Total Degree Distribution')
axes[1].hist(list(in_degrees.values()), bins=30, color='salmon')
axes[1].set_title('In-Degree Distribution')
axes[2].hist(list(out_degrees.values()), bins=30, color='lightgreen')
axes[2].set_title('Out-Degree Distribution')
plt.tight_layout()
plt.show()

top_hub = sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)[:10]
print('\nTop 10 TF hubs (by out-degree):')
for tf, deg in top_hub:
    sign = grn.edges[tf, list(grn.successors(tf))[0]].get('sign', 1) if deg > 0 else 0
    print(f'  {tf}: {deg} targets')

## 4. Visualize Core GRN Subgraph

In [ ]:
top_nodes = [n for n, _ in sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:30]]
subgraph = grn.subgraph(top_nodes)

pos = nx.spring_layout(subgraph, seed=42, k=2)
edge_colors = ['#E74C3C' if d.get('sign', 1) < 0 else '#2ECC71'
               for _, _, d in subgraph.edges(data=True)]
node_sizes = [300 + 30 * degrees.get(n, 0) for n in subgraph.nodes()]

plt.figure(figsize=(14, 10))
nx.draw_networkx(
    subgraph, pos=pos, node_size=node_sizes,
    edge_color=edge_colors, arrows=True,
    font_size=8, node_color='#AED6F1', alpha=0.9
)
plt.title('Core GRN (top 30 hub TFs)')
plt.axis('off')
plt.tight_layout()
plt.show()

## 5. Cancer vs Normal TF Differential Activity

In [ ]:
import pandas as pd

cancer_cells = adata[adata.obs['cell_state'] == 'cancer']
normal_cells = adata[adata.obs['cell_state'] == 'normal']

tf_nodes = [n for n in grn.nodes() if n in adata.var_names]
records = []
for tf in tf_nodes:
    if tf in cancer_cells.var_names and tf in normal_cells.var_names:
        cancer_expr = float(cancer_cells[:, tf].X.mean())
        normal_expr = float(normal_cells[:, tf].X.mean())
        records.append({'tf': tf, 'cancer_expr': cancer_expr,
                        'normal_expr': normal_expr,
                        'delta': cancer_expr - normal_expr})

df = pd.DataFrame(records).sort_values('delta', ascending=False)
print('Top upregulated TFs in cancer:')
print(df.head(10).to_string(index=False))
print('\nTop downregulated TFs in cancer:')
print(df.tail(10).to_string(index=False))

## 6. Save GRN

In [ ]:
import pickle
out_path = DATA_DIR / 'processed/networks/colorectal_GSE132465_grn.pkl'
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'wb') as f:
    pickle.dump(grn, f)
print(f'Saved GRN to {out_path}')